In [4]:
import os
import cv2
import numpy as np
import pandas as pd
from glob import glob

# ── CONFIG ────────────────────────────────────────────────────────────────
site = "nemours"
cam_id = "192_168_1_12"

base_dir      = f"/Users/mateo/pyronear/vision/smoke-localization/site_data/{site}"
ref_folder    = os.path.join(base_dir, "ref_images", cam_id)
pose_folder   = os.path.join(base_dir, "captured_poses", cam_id)
csv_path      = os.path.join(base_dir, f"df_ref_images_{cam_id}.csv")
output_csv    = os.path.join(base_dir, f"df_pose_azimuths_{cam_id}.csv")

# ── UTILITIES ────────────────────────────────────────────────────────────
def estimate_dx_angle(img1, img2, fov_deg=54.2, match_count=100):
    """Return angular offset between two cv2 images using ORB→BFMatcher."""
    H, W = img1.shape[:2]
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    orb = cv2.ORB_create(nfeatures=1000)
    kp1, des1 = orb.detectAndCompute(gray1, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)
    if des1 is None or des2 is None:
        return None

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = sorted(bf.match(des1, des2), key=lambda m: m.distance)[:match_count]

    # filter out matches near center to avoid degenerate
    margin_x, margin_y = W*0.2, H*0.1
    dxs = []
    for m in matches:
        x1, y1 = kp1[m.queryIdx].pt
        x2, y2 = kp2[m.trainIdx].pt
        if (abs(x1 - W/2) > margin_x
            and margin_y < y1 < H - margin_y
            and margin_y < y2 < H - margin_y):
            dxs.append(x2 - x1)
    if not dxs:
        return None

    mean_dx = np.mean(dxs)
    fov_rad = np.radians(fov_deg)
    angle_rad = 2 * np.arctan((mean_dx * np.tan(fov_rad/2)) / W)
    return np.degrees(angle_rad)

# ── LOAD REFERENCE AZIMUTHS ───────────────────────────────────────────────
df_ref = pd.read_csv(csv_path)
df_ref['azimut_estime'] = pd.to_numeric(df_ref['azimut_estime'], errors='coerce')

# build full paths in the same order as df_ref
ref_paths = [os.path.join(ref_folder, fn) for fn in df_ref['image']]

# ── PRECOMPUTE REFERENCE DESCRIPTORS ────────────────────────────────────
orb   = cv2.ORB_create(nfeatures=1000)
bf    = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
refs  = []
for p in ref_paths:
    img = cv2.imread(p)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kp, des = orb.detectAndCompute(gray, None)
    refs.append({'path': p, 'kp': kp, 'des': des})

# ── PROCESS POSE IMAGES ─────────────────────────────────────────────────
results = []
pose_paths = sorted(glob(os.path.join(pose_folder, "*.*")))

for pose_p in pose_paths:
    pose_img = cv2.imread(pose_p)
    gray_p   = cv2.cvtColor(pose_img, cv2.COLOR_BGR2GRAY)
    kp_p, des_p = orb.detectAndCompute(gray_p, None)
    if des_p is None:
        print(f"⚠️ no keypoints in {pose_p}, skipping")
        continue

    # find best reference by match count
    best = None
    best_n = 0
    for r in refs:
        if r['des'] is None:
            continue
        m = bf.match(r['des'], des_p)
        if len(m) > best_n:
            best_n = len(m)
            best = r

    if best is None:
        print(f"⚠️ no match for {pose_p}, skipping")
        continue

    # compute delta angle between best.ref and this pose
    ref_img = cv2.imread(best['path'])
    delta   = estimate_dx_angle(ref_img, pose_img)
    if delta is None:
        print(f"⚠️ could not compute delta for {pose_p}, skipping")
        continue

    # lookup reference azimuth
    ref_name = os.path.basename(best['path'])
    ref_az   = float(df_ref.loc[df_ref['image']==ref_name, 'azimut_estime'].iloc[0])

    # compute pose azimuth
    pose_az  = (ref_az + delta) % 360

    results.append({
        'pose_image':        os.path.basename(pose_p),
        'matched_ref_image': ref_name,
        'delta_angle':       delta,
        'ref_azimuth':       ref_az,
        'pose_azimuth':      pose_az
    })

# ── SAVE RESULTS ────────────────────────────────────────────────────────
df_out = pd.DataFrame(results)
df_out.to_csv(output_csv, index=False)
print(f"✅ Saved pose azimuths to {output_csv}")


✅ Saved pose azimuths to /Users/mateo/pyronear/vision/smoke-localization/site_data/nemours/df_pose_azimuths_192_168_1_12.csv
